In [54]:
import kagglehub
import pandas as pd
import os

# Download dataset
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("Dataset downloaded to:", path)

# Find the CSV file (usually named 'spam.csv' or similar)
for file in os.listdir(path):
    if file.endswith('.csv'):
        csv_path = os.path.join(path, file)
        print("CSV file found:", csv_path)
        break

# Load the CSV
df = pd.read_csv(csv_path, encoding='latin-1')
print("First 5 rows:")
print(df.head())

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Dataset downloaded to: /kaggle/input/sms-spam-collection-dataset
CSV file found: /kaggle/input/sms-spam-collection-dataset/spam.csv
First 5 rows:
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [55]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib
import kagglehub
import os

nltk.download('stopwords', quiet=True)

True

In [56]:
# ------------------------------------------------------------
# 1. Load the SMS Spam Collection dataset
# ------------------------------------------------------------
print("Step 1: Downloading dataset...")
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("Dataset path:", path)

# Locate the CSV file
csv_file = None
for file in os.listdir(path):
    if file.endswith('.csv'):
        csv_file = os.path.join(path, file)
        break

if csv_file is None:
    raise FileNotFoundError("No CSV file found in the downloaded dataset")

print("Loading CSV:", csv_file)
df = pd.read_csv(csv_file, encoding='latin-1')

# The SMS dataset has columns: v1 (label), v2 (message)
# Drop any extra unnamed columns
df = df[['v1', 'v2']]
df.columns = ['label', 'text']   # rename to match our code

# Convert labels: spam -> 1, ham -> 0
df['label'] = df['label'].map({'spam': 1, 'ham': 0})

print(f"Total samples: {len(df)}")
print(f"Spam count: {df['label'].sum()} | Ham count: {len(df) - df['label'].sum()}")
print("Sample data:")
print(df.head())


Step 1: Downloading dataset...
Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Dataset path: /kaggle/input/sms-spam-collection-dataset
Loading CSV: /kaggle/input/sms-spam-collection-dataset/spam.csv
Total samples: 5572
Spam count: 747 | Ham count: 4825
Sample data:
   label                                               text
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


In [57]:
# ------------------------------------------------------------
# 2. Text preprocessing
# ------------------------------------------------------------

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

X = df['text'].apply(preprocess_text)
y = df['label']

In [58]:
# ------------------------------------------------------------
# 3. Train-test split
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")


Train size: 4457, Test size: 1115


In [59]:
# ------------------------------------------------------------
# 4. TF-IDF vectorization
# ------------------------------------------------------------
print("Step 3: Vectorizing with TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf)
print(X_test_tfidf)


Step 3: Vectorizing with TF-IDF...
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 34711 stored elements and shape (4457, 5000)>
  Coords	Values
  (0, 1627)	0.5141642092877227
  (0, 3056)	0.8576917662467858
  (1, 4893)	0.4838438501871995
  (1, 4808)	0.4236741004393276
  (1, 4875)	0.5935406590023804
  (1, 1675)	0.4838438501871995
  (2, 3151)	0.24772381829046913
  (2, 2285)	0.5332607129444951
  (2, 2275)	0.4186524656374954
  (2, 4234)	0.5332607129444951
  (2, 167)	0.4411678217267143
  (3, 4421)	0.3432939324327316
  (3, 194)	0.3116026949213549
  (3, 1131)	0.3030576245892747
  (3, 1711)	0.2894508759364373
  (3, 31)	0.3928725559389737
  (3, 4877)	0.3116026949213549
  (3, 1153)	0.3975057130260467
  (3, 4892)	0.2865670993760587
  (3, 855)	0.3432939324327316
  (4, 1504)	0.5130904847889537
  (4, 4896)	0.45610682962786936
  (4, 4965)	0.5316360708058425
  (4, 4335)	0.4960522176182416
  (5, 4335)	0.2591652972987682
  :	:
  (4455, 4827)	0.23558663626275766
  (4455, 578)	0.209353242579

In [60]:
# ------------------------------------------------------------
# 5. Train Naive Bayes classifier
# ------------------------------------------------------------
print(" Training Multinomial Naive Bayes...")
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)


 Training Multinomial Naive Bayes...


MultinomialNB()

In [61]:
# ------------------------------------------------------------
# 6. Evaluate
# ------------------------------------------------------------
y_pred = model.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\n=== Evaluation Results ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)



=== Evaluation Results ===
Accuracy:  0.9641
Precision: 0.9910
Recall:    0.7383
F1-score:  0.8462
Confusion Matrix:
[[965   1]
 [ 39 110]]


In [62]:
# ------------------------------------------------------------
# 7. Save model and vectorizer
# ------------------------------------------------------------
joblib.dump(model, 'spam_detector_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("\nModel and vectorizer saved successfully.")


Model and vectorizer saved successfully.


In [63]:
# ------------------------------------------------------------
# 8. Prediction example
# ------------------------------------------------------------
def predict_email(email_text):
    model = joblib.load('spam_detector_model.pkl')
    tfidf = joblib.load('tfidf_vectorizer.pkl')
    cleaned = preprocess_text(email_text)
    vec = tfidf.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    return pred, prob

new_email = """
Congratulations! You've won a $1000 Amazon gift card.
Click here to claim your prize now: http://fake-link.com
"""
pred, prob = predict_email(new_email)
print("\n--- Example Prediction ---")
print(f"Email text: {new_email[:100]}...")
print(f"Prediction: {'SPAM' if pred == 1 else 'HAM'} (probability: {prob[pred]:.2f})")


--- Example Prediction ---
Email text: 
Congratulations! You've won a $1000 Amazon gift card. 
Click here to claim your prize now: http://f...
Prediction: SPAM (probability: 0.86)
